# Jour 2 · Mini-challenge de forecasting

## Objectifs

- reconstruire un protocole complet sans recopier la démonstration ;
- comparer une baseline et Prophet sur le même futur ;
- contrôler chaque étape avant de continuer ;
- interpréter les erreurs au lieu de regarder uniquement un score ;
- formuler une recommandation compréhensible par une équipe métier.

## Mission

L'équipe qui exploite le bâtiment veut anticiper la **température horaire des sept derniers jours** du jeu de données. Elle hésite entre une règle saisonnière très simple et un modèle Prophet.

Votre rôle n'est pas seulement d'obtenir le score le plus bas. Vous devez construire une comparaison honnête, repérer les moments difficiles à prévoir et recommander une approche exploitable.

Toutes les décisions doivent être prises avec le passé. Les valeurs du test servent uniquement à l'évaluation finale.

![Cinq étapes d'un protocole de forecasting honnête](../assets/jour_02/04_pipeline_forecasting.png)

*Découper, construire une baseline, entraîner Prophet, mesurer puis expliquer les erreurs : chaque étape répond à une question différente.*

## Livrables attendus

À la fin du challenge, votre notebook doit contenir :

1. les bornes chronologiques de train et test ;
2. une baseline saisonnière hebdomadaire ;
3. une prévision Prophet sur les mêmes 168 heures ;
4. un tableau MAE/RMSE comparant les deux méthodes ;
5. un graphique réel/baseline/Prophet ;
6. les cinq plus grandes erreurs du meilleur modèle ;
7. une recommandation de quelques phrases destinée à l'équipe HVAC.

Une bonne réponse explique le **pourquoi** des décisions et ne se limite pas à du code qui s'exécute.

## Règles du challenge

- Ne mélangez jamais les lignes avant le découpage.
- Utilisez les mêmes timestamps de test pour les deux méthodes.
- N'ajustez pas un paramètre après avoir regardé les erreurs du test sans le signaler.
- Donnez une unité aux métriques et aux axes.
- Conservez les contrôles avec assert : ils documentent vos hypothèses.
- Si un résultat semble surprenant, observez les données avant de modifier le modèle.

## Point de départ

La cellule suivante charge le fichier propre et fabrique une température horaire. Le reste du protocole vous appartient. Vérifiez d'abord la période et l'absence de valeurs manquantes.

In [1]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"
plt.style.use("seaborn-v0_8-whitegrid")

data = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_clean.csv")
data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)
temperature = (
    data.set_index("timestamp")["temperature_c"]
    .resample("1h")
    .mean()
    .rename("temperature_c")
)

print(temperature.index.min(), "→", temperature.index.max())
print(len(temperature), "heures |", temperature.isna().sum(), "valeurs manquantes")

C:\Users\bstorm_user\Desktop\time series\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-07-01 00:00:00+00:00 → 2026-08-11 23:00:00+00:00
1008 heures | 0 valeurs manquantes


## Étape 1 — découper chronologiquement

Placez les sept derniers jours dans test et tout le reste dans train. Affichez pour chaque partie : nombre de lignes, première date et dernière date.

Ajoutez deux assertions :

- le dernier timestamp de train est antérieur au premier de test ;
- le test contient exactement 168 heures.

**Indice :** calculez une frontière à partir du timestamp maximal moins sept jours.

In [2]:
# Écrivez votre code ici.
pass

### Point de contrôle 1

Avant de continuer, vous devez pouvoir répondre oui à chaque question :

- train se trouve-t-il entièrement avant test ?
- les deux séries sont-elles triées ?
- leurs index ne se chevauchent-ils pas ?
- le test représente-t-il bien sept journées complètes ?

Si un contrôle échoue, corrigez le découpage maintenant. Toutes les étapes suivantes en dépendent.

## Étape 2 — construire la baseline saisonnière

Répétez les 168 dernières heures de train pour prévoir test. Créez une Series nommée seasonal_naive avec exactement le même index que test.

Calculez ensuite :

- MAE en degrés Celsius ;
- RMSE en degrés Celsius.

**Contrôles attendus :** la baseline contient 168 valeurs, aucun NaN et son index est identique à celui de test.

**Question :** pourquoi une période hebdomadaire paraît-elle plus raisonnable qu'une simple persistance pour la température d'un bâtiment ?

In [3]:
# Écrivez votre code ici.
pass

### Point de contrôle 2

Affichez les quatre premières lignes d'un tableau réel/baseline. Vérifiez visuellement que les timestamps sont alignés.

Une baseline saisonnière peut avoir une erreur faible tout en ratant un changement de météo ou de consigne. Notez une situation réelle qui rendrait la semaine précédente peu représentative.

## Étape 3 — préparer et entraîner Prophet

Transformez train en tableau ds/y, puis retirez le fuseau de ds. Entraînez Prophet avec :

- saisonnalité journalière activée ;
- saisonnalité hebdomadaire activée ;
- saisonnalité annuelle désactivée ;
- uncertainty_samples égal à zéro.

Ne fournissez jamais test à la méthode fit.

**Contrôles attendus :** prophet_train possède seulement ds et y ; sa dernière date correspond à la fin de train ; y ne contient aucune valeur manquante.

In [4]:
# Écrivez votre code ici.
pass

### Point de contrôle 3

Expliquez avec vos propres mots ce que le modèle vient d'apprendre. Votre réponse doit mentionner tendance, cycle journalier et cycle hebdomadaire, sans prétendre que Prophet connaît la météo ou l'état physique de la machine.

## Étape 4 — prévoir exactement le test

Créez le tableau future à partir des timestamps de test sans fuseau, appelez predict, puis placez yhat dans une Series predicted indexée comme test.

Calculez MAE et RMSE de Prophet.

**Contrôles attendus :** predicted et test ont la même longueur et le même index ; les métriques portent l'unité °C ; aucune valeur réelle du test n'a été transmise à fit ou predict.

In [5]:
# Écrivez votre code ici.
pass

## Étape 5 — comparer équitablement

Construisez un DataFrame avec une ligne par méthode et deux colonnes : MAE (°C) et RMSE (°C). Tracez ensuite réel, baseline et Prophet sur le même graphique.

Votre lecture doit répondre à quatre questions :

1. Quelle méthode a la meilleure MAE ?
2. Le classement est-il identique avec la RMSE ?
3. Quelle courbe reproduit le mieux le rythme journalier ?
4. Le gain éventuel de Prophet paraît-il suffisant pour justifier sa complexité ?

Un tableau sans interprétation ne termine pas cette étape.

In [6]:
# Écrivez votre code ici.
pass

### Point de contrôle 4

Le graphique doit contenir un titre, une légende, un axe temporel et l'unité °C. Zoomez mentalement sur plusieurs situations :

- les pics de température ;
- les creux nocturnes ;
- le début et la fin du test ;
- une zone où les deux méthodes se trompent ensemble.

Si les courbes semblent décalées, vérifiez d'abord les index plutôt que d'accuser le modèle.

## Étape 6 — expliquer les plus grandes erreurs

Choisissez le modèle qui possède la plus petite MAE. Construisez un tableau avec réel, prévision, résidu signé et erreur absolue, puis affichez ses cinq plus grandes erreurs.

Pour chacune, observez le timestamp et le sens du résidu. Cherchez ensuite un motif : même heure, plusieurs heures consécutives, sous-estimations ou surestimations.

**Contrôles attendus :** le tableau possède 168 lignes ; l'erreur absolue est positive ; les cinq lignes sont triées de la plus grande à la plus petite erreur.

In [7]:
# Écrivez votre code ici.
pass

## Étape 7 — formuler une recommandation

Rédigez cinq à huit phrases destinées à l'équipe HVAC. Votre recommandation doit contenir :

- la méthode que vous retiendriez aujourd'hui ;
- les deux métriques, avec leur unité ;
- l'importance réelle ou non de l'écart entre les méthodes ;
- une faiblesse observée sur la courbe ou dans les grandes erreurs ;
- une donnée supplémentaire qui pourrait améliorer la prévision ;
- une précaution avant la mise en production.

Évitez « Prophet est meilleur car son score est plus petit ». Traduisez le résultat en choix technique et métier.

In [8]:
# Rédigez ici votre recommandation sous forme de commentaires.
pass

## Questions de recul

Répondez oralement ou dans une cellule Markdown :

1. Le meilleur modèle est-il forcément le même avec MAE et RMSE ?
2. Pourquoi une seule semaine de test limite-t-elle la conclusion ?
3. Que se passerait-il lors d'une vague de chaleur absente de l'entraînement ?
4. Quelle variable externe ajouteriez-vous en priorité ?
5. Comment détecter qu'un modèle autrefois correct se dégrade en production ?
6. Une grande erreur de prévision peut-elle servir à autre chose qu'à mesurer la qualité du modèle ?

La dernière question prépare directement le notebook précédent sur les alertes et la journée consacrée à la détection d'anomalies.

## Auto-évaluation

Cochez mentalement une affirmation uniquement si vous pouvez l'expliquer sans relire le notebook :

- [ ] Je sais pourquoi le test doit être postérieur à l'entraînement.
- [ ] Je sais construire et justifier une baseline saisonnière.
- [ ] Je sais expliquer la différence entre MAE et RMSE.
- [ ] Je sais préparer les colonnes ds et y pour Prophet.
- [ ] Je sais comparer deux méthodes sur le même futur.
- [ ] Je sais retrouver les timestamps responsables des plus grandes erreurs.
- [ ] Je sais transformer un résultat technique en recommandation métier.

Si une case reste incertaine, revenez à l'étape correspondante avant de passer au Jour 3.

## Synthèse du Jour 2

~~~text
question métier
    ↓
cible et horizon
    ↓
découpage passé / futur
    ↓
baseline simple
    ↓
modèle Prophet
    ↓
MAE + RMSE + graphique
    ↓
analyse des résidus
    ↓
décision et surveillance
~~~

Demain, nous n'utiliserons plus seulement l'erreur pour juger une prévision. Nous l'utiliserons, avec d'autres méthodes, pour repérer des comportements inhabituels et prioriser une investigation de maintenance.